# ID4 - Read and Partition Benchmark

This notebook reads the four format outputs from ID3, runs one fixed aggregate query, and writes partition-control outputs for comparison.

## 1. Setup

Use the same SparkSession style as ID3. Adjust paths only if your ID3 output is elsewhere.

In [ ]:
from pathlib import Path
import csv
import os
import statistics
import time

from pyspark.sql import SparkSession, functions as F

def find_project_root(start):
    start = Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / "requirements.txt").exists() and (path / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root from current directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
ID3_OUTPUT = PROJECT_ROOT / "output" / "format_benchmark"
ID4_OUTPUT = PROJECT_ROOT / "output" / "partition_test"

FORMATS = ["csv", "json", "parquet", "orc"]
OFFICIAL_RUNS = 3

spark = (
    SparkSession.builder
    .appName("ReadPartitionBenchmarkID4")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

ID4_OUTPUT.mkdir(parents=True, exist_ok=True)
print("ID3 output:", ID3_OUTPUT)
print("ID4 output:", ID4_OUTPUT)

## 2. Readers

Use the Parquet schema as the canonical schema. CSV and JSON are read with this schema so schema inference is not timed for only those formats.

In [ ]:
for fmt in FORMATS:
    path = ID3_OUTPUT / fmt
    if not path.exists():
        raise FileNotFoundError(f"Missing ID3 output for {fmt}: {path}")

canonical_schema = spark.read.parquet(str(ID3_OUTPUT / "parquet")).schema

def read_format(fmt):
    path = str(ID3_OUTPUT / fmt)
    if fmt == "csv":
        return spark.read.option("header", True).schema(canonical_schema).csv(path)
    if fmt == "json":
        return spark.read.schema(canonical_schema).json(path)
    if fmt == "parquet":
        return spark.read.parquet(path)
    if fmt == "orc":
        return spark.read.orc(path)
    raise ValueError(fmt)

def run_query(df):
    return (
        df.groupBy("Carrier")
        .agg(F.avg("ArrDelay").alias("avg_arr_delay"))
        .orderBy("Carrier")
        .collect()
    )

## 3. Read Query Benchmark

Each format gets one warm-up run, then three official runs. Timing wraps the action (`collect`).

In [ ]:
read_runs = []
read_summary = []

for fmt in FORMATS:
    df = read_format(fmt)
    run_query(df)  # warm-up, excluded

    times = []
    for run_number in range(1, OFFICIAL_RUNS + 1):
        start = time.perf_counter()
        rows = run_query(df)
        elapsed = time.perf_counter() - start
        times.append(elapsed)

        read_runs.append({
            "format": fmt,
            "run_number": run_number,
            "read_time_sec": elapsed,
            "result_rows": len(rows),
        })
        print(f"{fmt.upper():8} run {run_number}: {elapsed:.3f}s, rows={len(rows)}")

    read_summary.append({
        "format": fmt,
        "official_runs": OFFICIAL_RUNS,
        "median_read_time_sec": statistics.median(times),
    })

read_summary

In [ ]:
def write_csv(path, rows, fieldnames):
    with open(path, "w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

write_csv(
    ID4_OUTPUT / "read_benchmark_runs.csv",
    read_runs,
    ["format", "run_number", "read_time_sec", "result_rows"],
)
write_csv(
    ID4_OUTPUT / "read_benchmark_summary.csv",
    read_summary,
    ["format", "official_runs", "median_read_time_sec"],
)

print("Saved read benchmark CSV files.")

## 4. Partition Experiments

Write three separate Parquet outputs: `repartition(20)`, `coalesce(2)`, and directory partitioning by `Year`, `Month`.

In [ ]:
def part_file_stats(path):
    total_bytes = 0
    part_file_count = 0
    for root, _, files in os.walk(path):
        for filename in files:
            if filename.startswith("part-"):
                part_file_count += 1
                total_bytes += os.path.getsize(os.path.join(root, filename))
    avg_bytes = total_bytes / part_file_count if part_file_count else 0
    return part_file_count, total_bytes, avg_bytes

def timed_partition_write(name, df, output_path, writer_fn):
    start = time.perf_counter()
    writer_fn(df).write.mode("overwrite").parquet(str(output_path))
    elapsed = time.perf_counter() - start
    part_count, total_bytes, avg_bytes = part_file_stats(output_path)
    return {
        "experiment": name,
        "output_path": str(output_path),
        "write_time_sec": elapsed,
        "part_file_count": part_count,
        "size_bytes": total_bytes,
        "size_mb": total_bytes / (1024 ** 2),
        "avg_part_size_mb": avg_bytes / (1024 ** 2),
    }

base_df = read_format("parquet")

partition_results = [
    timed_partition_write(
        "repartition_20",
        base_df,
        ID4_OUTPUT / "repartition_20",
        lambda df: df.repartition(20),
    ),
    timed_partition_write(
        "coalesce_2",
        base_df,
        ID4_OUTPUT / "coalesce_2",
        lambda df: df.coalesce(2),
    ),
]

start = time.perf_counter()
partition_by_path = ID4_OUTPUT / "by_year_month"
base_df.write.mode("overwrite").partitionBy("Year", "Month").parquet(str(partition_by_path))
elapsed = time.perf_counter() - start
part_count, total_bytes, avg_bytes = part_file_stats(partition_by_path)
partition_results.append({
    "experiment": "partition_by_year_month",
    "output_path": str(partition_by_path),
    "write_time_sec": elapsed,
    "part_file_count": part_count,
    "size_bytes": total_bytes,
    "size_mb": total_bytes / (1024 ** 2),
    "avg_part_size_mb": avg_bytes / (1024 ** 2),
})

partition_results

In [ ]:
write_csv(
    ID4_OUTPUT / "partition_benchmark_summary.csv",
    partition_results,
    [
        "experiment",
        "output_path",
        "write_time_sec",
        "part_file_count",
        "size_bytes",
        "size_mb",
        "avg_part_size_mb",
    ],
)
print("Saved partition benchmark summary.")

## 5. Folder Layout Check

`partitionBy("Year", "Month")` creates directory partitions such as `Year=2015/Month=1/part-*.parquet`.

In [ ]:
def preview_tree(path, max_lines=40):
    path = Path(path)
    lines = []
    for root, dirs, files in os.walk(path):
        rel = Path(root).relative_to(path)
        depth = 0 if str(rel) == "." else len(rel.parts)
        if depth <= 3:
            indent = "  " * depth
            name = path.name if str(rel) == "." else Path(root).name
            lines.append(f"{indent}{name}/")
            for filename in sorted(files):
                if filename.startswith("part-"):
                    lines.append(f"{indent}  {filename}")
        if len(lines) >= max_lines:
            break
    return "\n".join(lines[:max_lines])

print(preview_tree(ID4_OUTPUT / "by_year_month"))

## 6. Close Spark

Stop Spark after all outputs are saved.

In [ ]:
spark.stop()
print("SparkSession stopped.")